
PakWheels Full Dataset Scraper
==============================
Collects ALL available data per listing:
  - Search card fields  (title, price, year, mileage, engine, transmission, fuel, location)
  - Detail page fields  (color, registered city, assembly, body type, last updated,
                         seller name, seller type, seller phone, description, all specs)
  - Images              (all gallery photos downloaded per listing)

Usage:
    pip install requests beautifulsoup4 pandas lxml
    python pakwheels_scraper.py

Output:
    pakwheels_vehicles.csv   — full dataset
    pakwheels_images/
        <listing_id>/
            01.jpg, 02.jpg, ...



In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import logging
import re
import os
import json
from urllib.parse import urljoin

In [2]:
# ── Logging ───────────────────────────────────────────────────────────────────
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler("scraper.log", encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)

# ── Config ────────────────────────────────────────────────────────────────────
BASE_URL   = "https://www.pakwheels.com"
SEARCH_URL = "https://www.pakwheels.com/used-cars/search/-/"
IMAGES_DIR = "pakwheels_images"
OUTPUT_CSV = "pakwheels_vehicles.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

PAGE_DELAY   = 2    # seconds between search-result pages
DETAIL_DELAY = 2    # seconds between detail page visits
IMAGE_DELAY  = 0.5  # seconds between image downloads
MAX_RETRIES  = 3
BACKOFF      = 2

In [3]:
# ── HTTP helper ───────────────────────────────────────────────────────────────
def get_page(url, params=None):
    """Fetch URL and return BeautifulSoup, with retries."""
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=15)
            r.raise_for_status()
            return BeautifulSoup(r.text, "lxml")
        except requests.HTTPError as e:
            code = e.response.status_code
            if code in (429, 503):
                wait = BACKOFF ** (attempt + 1)
                logger.warning(f"Rate limited ({code}). Waiting {wait}s…")
                time.sleep(wait)
            else:
                logger.error(f"HTTP {code} on {url}")
                return None
        except requests.RequestException as e:
            wait = BACKOFF ** (attempt + 1)
            logger.warning(f"Request error ({e}). Retry {attempt+1}/{MAX_RETRIES} in {wait}s")
            time.sleep(wait)
    logger.error(f"Gave up on {url} after {MAX_RETRIES} attempts.")
    return None

In [4]:
# ── Listing ID extraction ─────────────────────────────────────────────────────
def extract_listing_id(url):
    """
    Try multiple URL patterns PakWheels has used historically.
    Prints unmatched URLs so you can add a pattern if needed.
    """
    if not url:
        return None
    patterns = [
        r"/(\d{6,})/?$",           # ends with digits:  /1234567/ or /1234567
        r"-(\d{6,})\.html",        # slug-123456.html
        r"[?&]ad_id=(\d+)",        # query param
        r"/detail/[^/]+/(\d+)",    # /detail/model/123456
    ]
    for pattern in patterns:
        m = re.search(pattern, url)
        if m:
            return m.group(1)
    logger.debug(f"Could not extract listing_id from URL: {url}")
    return None


In [5]:
# ── Search-card parser ────────────────────────────────────────────────────────
def parse_card(card):
    """Extract fields visible on the search results page."""
    data = {}

    # Title + URL
    # Try multiple known selector variations
    title_tag = (
        card.select_one("a.car-name.ad-detail-path") or
        card.select_one("a.car-name") or
        card.select_one("h3.car-name a") or
        card.select_one("a[href*='/used-cars/']")
    )
    if title_tag:
        data["title"] = title_tag.get_text(strip=True)
        href = title_tag.get("href", "")
        data["url"] = urljoin(BASE_URL, href) if href else None
    else:
        data["title"] = None
        data["url"] = None

    data["listing_id"] = extract_listing_id(data["url"])

    # Price
    price_tag = (
        card.select_one("div.price-details") or
        card.select_one("strong.price") or
        card.select_one(".price")
    )
    data["price"] = price_tag.get_text(strip=True) if price_tag else None

    # Location
    loc_tag = (
        card.select_one("ul.search-vehicle-info-2 li.item-location") or
        card.select_one("li.item-location") or
        card.select_one(".location")
    )
    data["location"] = loc_tag.get_text(strip=True) if loc_tag else None

    # Specs row (year / mileage / engine / transmission / fuel)
    spec_items = card.select("ul.search-vehicle-info li")
    spec_texts  = [s.get_text(strip=True) for s in spec_items]

    data["year"]         = spec_texts[0] if len(spec_texts) > 0 else None
    data["mileage"]      = spec_texts[1] if len(spec_texts) > 1 else None
    data["engine_cc"]    = spec_texts[2] if len(spec_texts) > 2 else None
    data["transmission"] = spec_texts[3] if len(spec_texts) > 3 else None
    data["fuel_type"]    = spec_texts[4] if len(spec_texts) > 4 else None

    return data

In [6]:
# ── Detail-page parser ────────────────────────────────────────────────────────
def parse_detail(url):
    """
    Visit the individual listing page and extract ALL available fields.
    Returns a dict of extra fields to merge into the main record.
    """
    soup = get_page(url)
    if not soup:
        return {}

    detail = {}

    # ── Structured specs table (ul.car-feature-list or table) ────────────────
    # PakWheels renders specs as a <ul> with <li> pairs: label / value
    spec_rows = soup.select("ul.car-feature-list li") or soup.select("ul.list-unstyled li")
    for li in spec_rows:
        label_tag = li.select_one(".title") or li.select_one("span:first-child")
        value_tag = li.select_one(".detail") or li.select_one("span:last-child")
        if label_tag and value_tag:
            key = label_tag.get_text(strip=True).lower().replace(" ", "_").replace("/", "_")
            val = value_tag.get_text(strip=True)
            detail[f"spec_{key}"] = val

    # Fallback: grab specs from a table
    for row in soup.select("table tr"):
        cells = row.select("td")
        if len(cells) == 2:
            key = cells[0].get_text(strip=True).lower().replace(" ", "_")
            val = cells[1].get_text(strip=True)
            if key:
                detail[f"spec_{key}"] = val

    # ── Description ──────────────────────────────────────────────────────────
    desc_tag = (
        soup.select_one("div.car-detail-desc p") or
        soup.select_one(".ad-detail-desc") or
        soup.select_one("#description")
    )
    detail["description"] = desc_tag.get_text(strip=True) if desc_tag else None

    # ── Seller info ───────────────────────────────────────────────────────────
    seller_name = (
        soup.select_one(".seller-name") or
        soup.select_one(".dealer-name") or
        soup.select_one("span.name")
    )
    detail["seller_name"] = seller_name.get_text(strip=True) if seller_name else None

    seller_type = soup.select_one(".seller-type") or soup.select_one(".badge-seller")
    detail["seller_type"] = seller_type.get_text(strip=True) if seller_type else None

    # Phone (often hidden behind JS; grab data attribute if present)
    phone_tag = soup.select_one("[data-phone]") or soup.select_one(".seller-phone")
    if phone_tag:
        detail["seller_phone"] = phone_tag.get("data-phone") or phone_tag.get_text(strip=True)
    else:
        detail["seller_phone"] = None

    # ── Posted / updated date ─────────────────────────────────────────────────
    date_tag = (
        soup.select_one("span.date-an") or
        soup.select_one(".posted-date") or
        soup.select_one("time")
    )
    detail["posted_date"] = date_tag.get_text(strip=True) if date_tag else None

    # ── Gallery image URLs ────────────────────────────────────────────────────
    img_urls = []
    for img in soup.select("ul#car-gallery li img, .gallery img, .car-gallery img"):
        src = img.get("data-src") or img.get("data-lazy-src") or img.get("src", "")
        src = src.strip()
        if src.startswith("http") and src not in img_urls:
            img_urls.append(src)

    # Fallback: JSON-LD structured data sometimes has image array
    if not img_urls:
        for script in soup.select("script[type='application/ld+json']"):
            try:
                jd = json.loads(script.string or "")
                imgs = jd.get("image", [])
                if isinstance(imgs, str):
                    imgs = [imgs]
                img_urls.extend([i for i in imgs if i.startswith("http")])
            except Exception:
                pass

    detail["image_urls"] = "|".join(img_urls)   # pipe-separated in CSV
    detail["image_count"] = len(img_urls)

    return detail, img_urls

In [7]:
# ── Image downloader ──────────────────────────────────────────────────────────
def download_images(img_urls, listing_id):
    """Download all images for a listing into IMAGES_DIR/<listing_id>/"""
    if not img_urls:
        return 0

    folder = os.path.join(IMAGES_DIR, str(listing_id))
    os.makedirs(folder, exist_ok=True)

    saved = 0
    for idx, img_url in enumerate(img_urls):
        ext = img_url.split("?")[0].rsplit(".", 1)[-1]
        ext = ext if ext in ("jpg", "jpeg", "png", "webp") else "jpg"
        path = os.path.join(folder, f"{idx+1:02d}.{ext}")

        if os.path.exists(path):
            saved += 1
            continue

        try:
            r = requests.get(img_url, headers=HEADERS, timeout=15, stream=True)
            r.raise_for_status()
            with open(path, "wb") as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            saved += 1
        except Exception as e:
            logger.warning(f"Image download failed ({img_url}): {e}")

        time.sleep(IMAGE_DELAY)

    return saved

In [8]:
# ── Pagination ────────────────────────────────────────────────────────────────
def get_total_pages(soup):
    nums = [
        int(a.get_text(strip=True))
        for a in soup.select("ul.pagination li a")
        if a.get_text(strip=True).isdigit()
    ]
    return max(nums) if nums else 1

In [9]:
# ── Diagnosis helper ──────────────────────────────────────────────────────────
def diagnose(max_cards=3):
    """
    Run this first to verify selectors work on the live site.
    Prints raw URLs, IDs, and spec counts so you can confirm everything parses.
    """
    print("\n=== DIAGNOSIS MODE ===")
    soup = get_page(SEARCH_URL)
    if not soup:
        print("ERROR: Could not fetch search page. Check your internet connection.")
        return

    cards = soup.select("li.classified-listing")
    print(f"Cards found on page 1: {len(cards)}")

    for i, card in enumerate(cards[:max_cards]):
        d = parse_card(card)
        print(f"\n--- Card {i+1} ---")
        for k, v in d.items():
            print(f"  {k}: {v}")

    if cards:
        first = parse_card(cards[0])
        if first.get("url"):
            print(f"\nFetching detail page: {first['url']}")
            det, imgs = parse_detail(first["url"])
            print(f"Detail fields found: {len(det)}")
            print(f"Images found: {len(imgs)}")
            for k, v in list(det.items())[:15]:
                print(f"  {k}: {str(v)[:80]}")
    print("\n=== END DIAGNOSIS ===\n")

In [10]:
# ── Main scraper ──────────────────────────────────────────────────────────────
def scrape_pakwheels(
    max_pages=None,
    download_imgs=True,
    scrape_details=True,
    output_file=OUTPUT_CSV,
    resume=True,
):
    """
    Scrape all vehicle listings from PakWheels.

    Args:
        max_pages:       Cap number of pages (None = all).
        download_imgs:   Download all listing photos to disk.
        scrape_details:  Visit each listing's detail page for full specs.
        output_file:     CSV path to save results.
        resume:          Skip listings already saved in output_file.
    """
    os.makedirs(IMAGES_DIR, exist_ok=True)

    # Resume support — load already-scraped IDs
    scraped_ids = set()
    existing_df = None
    if resume and os.path.exists(output_file):
        existing_df = pd.read_csv(output_file, dtype=str)
        scraped_ids = set(existing_df["listing_id"].dropna().tolist())
        logger.info(f"Resuming — {len(scraped_ids)} listings already scraped.")

    all_listings = []
    page = 1
    effective_max = None

    logger.info("Starting PakWheels full scraper…")

    while True:
        url = f"{SEARCH_URL}?page={page}"
        logger.info(f"── Page {page} ── {url}")
        soup = get_page(url)

        if not soup:
            logger.warning("Empty response, stopping.")
            break

        if page == 1:
            total_pages = get_total_pages(soup)
            effective_max = min(max_pages, total_pages) if max_pages else total_pages
            logger.info(f"Total pages: {total_pages} | Will scrape: {effective_max}")

        cards = soup.select("li.classified-listing")
        if not cards:
            logger.info("No listings on this page — done.")
            break

        for card in cards:
            listing = parse_card(card)
            lid = listing.get("listing_id")

            # Skip if already scraped (resume mode)
            if lid and lid in scraped_ids:
                logger.debug(f"Skipping already-scraped listing {lid}")
                continue

            # Visit detail page
            if scrape_details and listing.get("url"):
                result = parse_detail(listing["url"])
                if isinstance(result, tuple):
                    detail_data, img_urls = result
                else:
                    detail_data, img_urls = result, []

                listing.update(detail_data)

                # Download images
                if download_imgs and lid and img_urls:
                    n = download_images(img_urls, lid)
                    listing["images_downloaded"] = n
                    logger.info(f"  [{lid}] {listing.get('title','?')} — {n} images saved")
                else:
                    listing["images_downloaded"] = 0

                time.sleep(DETAIL_DELAY)

            all_listings.append(listing)

        logger.info(f"  Page {page} done. New listings this session: {len(all_listings)}")

        # Incremental save every page
        if all_listings:
            new_df = pd.DataFrame(all_listings)
            if existing_df is not None:
                combined = pd.concat([existing_df, new_df], ignore_index=True)
            else:
                combined = new_df
            combined.drop_duplicates(subset="listing_id", keep="last", inplace=True)
            combined.to_csv(output_file, index=False, encoding="utf-8-sig")
            logger.info(f"  Saved {len(combined)} total listings to {output_file}")

        if page >= effective_max:
            break

        page += 1
        time.sleep(PAGE_DELAY)

    # Final save
    if all_listings:
        new_df = pd.DataFrame(all_listings)
        if existing_df is not None:
            final_df = pd.concat([existing_df, new_df], ignore_index=True)
        else:
            final_df = new_df
        final_df.drop_duplicates(subset="listing_id", keep="last", inplace=True)
        final_df.to_csv(output_file, index=False, encoding="utf-8-sig")
        logger.info(f"\n✓ Done! {len(final_df)} unique listings → {output_file}")
        return final_df

    elif existing_df is not None:
        return existing_df
    else:
        return pd.DataFrame()

In [11]:
# ── Entry point ───────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import argparse

    parser = argparse.ArgumentParser(description="PakWheels full dataset scraper")
    parser.add_argument("--diagnose",       action="store_true", help="Run diagnosis only")
    parser.add_argument("--max-pages",      type=int, default=None, help="Limit pages (default: all)")
    parser.add_argument("--no-images",      action="store_true",  help="Skip image downloads")
    parser.add_argument("--no-details",     action="store_true",  help="Skip detail page visits")
    parser.add_argument("--no-resume",      action="store_true",  help="Start fresh (ignore existing CSV)")
    parser.add_argument("--output",         default=OUTPUT_CSV,   help="Output CSV filename")
    args = parser.parse_args()

    if args.diagnose:
        diagnose()
    else:
        df = scrape_pakwheels(
            max_pages=args.max_pages,
            download_imgs=not args.no_images,
            scrape_details=not args.no_details,
            output_file=args.output,
            resume=not args.no_resume,
        )
        print(f"\nDataset shape: {df.shape}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(3))

usage: ipykernel_launcher.py [-h] [--diagnose] [--max-pages MAX_PAGES]
                             [--no-images] [--no-details] [--no-resume]
                             [--output OUTPUT]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\razap\AppData\Roaming\jupyter\runtime\kernel-v37f99e99e04b2e602ee66b778fd584b3f2a31d992.json


SystemExit: 2

C:\Users\razap\AppData\Roaming\Python\Python314\site-packages\IPython\core\interactiveshell.py:3709: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [12]:
# ── Run this cell in Jupyter ──────────────────────────────────────────────────

# STEP 1: Diagnose first (confirms selectors work)
diagnose()


=== DIAGNOSIS MODE ===
Cards found on page 1: 26

--- Card 1 ---
  title: KIA Picanto  2021 1.0 MT for Sale
  url: https://www.pakwheels.com/used-cars/kia-picanto-2021-for-sale-in-lahore-11248037
  listing_id: None
  price: PKR 25.95lacs
  location: None
  year: Lahore
  mileage: None
  engine_cc: None
  transmission: None
  fuel_type: None

--- Card 2 ---
  title: Toyota Fortuner  2013 2.7 VVTi for Sale
  url: https://www.pakwheels.com/used-cars/toyota-fortuner-2013-for-sale-in-islamabad-11223688
  listing_id: None
  price: PKR 66.5lacs
  location: None
  year: Islamabad
  mileage: None
  engine_cc: None
  transmission: None
  fuel_type: None

--- Card 3 ---
  title: Toyota Yaris Sedan  2026 GLI CVT 1.3 for Sale
  url: https://www.pakwheels.com/used-cars/toyota-yaris-sedan-2026-for-sale-in-karachi-11044350
  listing_id: None
  price: PKR 50.45lacs
  location: None
  year: Karachi
  mileage: None
  engine_cc: None
  transmission: None
  fuel_type: None

Fetching detail page: https://w

In [19]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import logging
import re
import os
import json
from urllib.parse import urljoin

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

BASE_URL   = "https://www.pakwheels.com"
SEARCH_URL = "https://www.pakwheels.com/used-cars/search/-/"
IMAGES_DIR = "pakwheels_images"
OUTPUT_CSV = "pakwheels_vehicles.csv"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

PAGE_DELAY   = 2
DETAIL_DELAY = 2
IMAGE_DELAY  = 0.5
MAX_RETRIES  = 3
BACKOFF      = 2


def get_page(url, params=None):
    for attempt in range(MAX_RETRIES):
        try:
            r = requests.get(url, headers=HEADERS, params=params, timeout=15)
            r.raise_for_status()
            return BeautifulSoup(r.text, "html.parser")
        except requests.HTTPError as e:
            code = e.response.status_code
            if code in (429, 503):
                wait = BACKOFF ** (attempt + 1)
                logger.warning(f"Rate limited ({code}). Waiting {wait}s…")
                time.sleep(wait)
            else:
                logger.error(f"HTTP {code} on {url}")
                return None
        except requests.RequestException as e:
            wait = BACKOFF ** (attempt + 1)
            logger.warning(f"Request error ({e}). Retry {attempt+1}/{MAX_RETRIES} in {wait}s")
            time.sleep(wait)
    return None


def extract_listing_id(url):
    """Matches URLs like: /used-cars/kia-picanto-2021-for-sale-in-lahore-11248037"""
    if not url:
        return None
    m = re.search(r"-(\d{6,})$", url.rstrip("/"))
    return m.group(1) if m else None

def parse_card(card):
    data = {}

    # Title + URL
    title_tag = card.select_one("div.search-title a") or card.select_one("a.car-name")
    data["title"] = title_tag.get_text(strip=True) if title_tag else None
    href = title_tag.get("href", "") if title_tag else ""
    data["url"] = urljoin(BASE_URL, href) if href else None
    data["listing_id"] = extract_listing_id(data["url"])

    # Price
    price_tag = card.select_one("div.price-details")
    data["price"] = price_tag.get_text(strip=True) if price_tag else None

    # All specs live in div.col-md-12.grid-date > ul > li (no classes, positional)
    spec_lis = card.select("div.col-md-12.grid-date li")
    spec_texts = [li.get_text(strip=True) for li in spec_lis]

    # Order from diagnosis: city, year, mileage, fuel, engine_cc, transmission
    data["location"]     = spec_texts[0] if len(spec_texts) > 0 else None
    data["year"]         = spec_texts[1] if len(spec_texts) > 1 else None
    data["mileage"]      = spec_texts[2] if len(spec_texts) > 2 else None
    data["fuel_type"]    = spec_texts[3] if len(spec_texts) > 3 else None
    data["engine_cc"]    = spec_texts[4] if len(spec_texts) > 4 else None
    data["transmission"] = spec_texts[5] if len(spec_texts) > 5 else None

    # Last updated date
    dated = card.select_one("div.dated")
    data["last_updated"] = dated.get_text(strip=True) if dated else None

    return data


def parse_detail(url):
    soup = get_page(url)
    if not soup:
        return {}, []

    detail = {}

    # Specs — diagnosis shows li.ad-data elements are labels,
    # and their sibling/next text node or adjacent element is the value
    # Structure: <li class="ad-data">Label<span>Value</span></li>
    for li in soup.select("li.ad-data"):
        full_text = li.get_text(separator="|", strip=True)
        parts = [p.strip() for p in full_text.split("|") if p.strip()]
        if len(parts) >= 2:
            key = re.sub(r"[^a-z0-9]+", "_", parts[0].lower()).strip("_")
            detail[f"spec_{key}"] = parts[1]
        elif len(parts) == 1:
            # value may be in next sibling
            nxt = li.find_next_sibling("li", class_="ad-data-value") or li.find_next_sibling()
            if nxt:
                key = re.sub(r"[^a-z0-9]+", "_", parts[0].lower()).strip("_")
                detail[f"spec_{key}"] = nxt.get_text(strip=True)

    # Features checklist (the ul.car-feature-list — these are checkboxes, not key-value)
    features = [li.get_text(strip=True) for li in soup.select("ul.car-feature-list li") if li.get_text(strip=True)]
    detail["features"] = ", ".join(features) if features else None

    # Description
    desc = (
        soup.select_one("p.detail-desc") or
        soup.select_one("div.car-detail-desc") or
        soup.select_one("#car-description p") or
        soup.select_one(".ad-desc")
    )
    detail["description"] = desc.get_text(strip=True) if desc else None

    # Seller name
    seller = soup.select_one("span.member") or soup.select_one(".seller-name")
    detail["seller_info"] = seller.get_text(strip=True) if seller else None

    # Phone — shown as "032XXXXXXXXShow Phone Number", extract digits
    phone_span = soup.select_one("span.fs22")
    if phone_span:
        raw = phone_span.get_text(strip=True)
        # strip the "Show Phone Number" suffix
        detail["seller_phone"] = raw.replace("Show Phone Number", "").strip()
    else:
        detail["seller_phone"] = None

    # Posted/updated date
    date_tag = soup.select_one("span.time") or soup.select_one(".posted-date")
    detail["posted_date"] = date_tag.get_text(strip=True) if date_tag else None

    # Images
    img_urls = []
    for img in soup.select("ul#car-gallery li img, .gallery img"):
        src = img.get("data-src") or img.get("data-lazy-src") or img.get("data-original") or img.get("src", "")
        src = src.strip()
        if src.startswith("http") and "placeholder" not in src and src not in img_urls:
            img_urls.append(src)

    # JSON-LD fallback
    if not img_urls:
        for script in soup.select("script[type='application/ld+json']"):
            try:
                jd = json.loads(script.string or "")
                imgs = jd.get("image", [])
                if isinstance(imgs, str):
                    imgs = [imgs]
                img_urls.extend([i for i in imgs if isinstance(i, str) and i.startswith("http")])
            except Exception:
                pass

    # og:image fallback
    if not img_urls:
        og = soup.find("meta", property="og:image")
        if og and og.get("content", "").startswith("http"):
            img_urls.append(og["content"])

    detail["image_urls"]  = "|".join(img_urls)
    detail["image_count"] = len(img_urls)

    return detail, img_urls

def download_images(img_urls, listing_id):
    if not img_urls:
        return 0
    folder = os.path.join(IMAGES_DIR, str(listing_id))
    os.makedirs(folder, exist_ok=True)
    saved = 0
    for idx, img_url in enumerate(img_urls):
        ext = img_url.split("?")[0].rsplit(".", 1)[-1]
        ext = ext if ext in ("jpg", "jpeg", "png", "webp") else "jpg"
        path = os.path.join(folder, f"{idx+1:02d}.{ext}")
        if os.path.exists(path):
            saved += 1
            continue
        try:
            r = requests.get(img_url, headers=HEADERS, timeout=15, stream=True)
            r.raise_for_status()
            with open(path, "wb") as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            saved += 1
        except Exception as e:
            logger.warning(f"Image failed ({img_url}): {e}")
        time.sleep(IMAGE_DELAY)
    return saved


def get_total_pages(soup):
    nums = [
        int(a.get_text(strip=True))
        for a in soup.select("ul.pagination li a")
        if a.get_text(strip=True).isdigit()
    ]
    return max(nums) if nums else 1


def diagnose(max_cards=3):
    print("\n=== DIAGNOSIS MODE ===")
    soup = get_page(SEARCH_URL)
    if not soup:
        print("ERROR: Could not fetch page.")
        return
    cards = soup.select("li.classified-listing")
    print(f"Cards found: {len(cards)}")
    for i, card in enumerate(cards[:max_cards]):
        d = parse_card(card)
        print(f"\n--- Card {i+1} ---")
        for k, v in d.items():
            print(f"  {k}: {v}")
    if cards:
        first = parse_card(cards[0])
        if first.get("url"):
            print(f"\nFetching detail: {first['url']}")
            det, imgs = parse_detail(first["url"])
            print(f"Detail fields: {len(det)} | Images: {len(imgs)}")
            for k, v in list(det.items())[:20]:
                print(f"  {k}: {str(v)[:100]}")
    print("=== END DIAGNOSIS ===\n")


def scrape_pakwheels(max_pages=None, download_imgs=True, scrape_details=True,
                     output_file=OUTPUT_CSV, resume=True):
    os.makedirs(IMAGES_DIR, exist_ok=True)

    scraped_ids = set()
    existing_df = None
    if resume and os.path.exists(output_file):
        existing_df = pd.read_csv(output_file, dtype=str)
        scraped_ids = set(existing_df["listing_id"].dropna().tolist())
        logger.info(f"Resuming — {len(scraped_ids)} listings already scraped.")

    all_listings = []
    page = 1
    effective_max = None

    while True:
        url = f"{SEARCH_URL}?page={page}"
        logger.info(f"── Page {page}/{effective_max or '?'} ──")
        soup = get_page(url)
        if not soup:
            break

        if page == 1:
            total_pages = get_total_pages(soup)
            effective_max = min(max_pages, total_pages) if max_pages else total_pages
            logger.info(f"Total pages: {total_pages} | Scraping: {effective_max}")

        cards = soup.select("li.classified-listing")
        if not cards:
            break

        for card in cards:
            listing = parse_card(card)
            lid = listing.get("listing_id")

            if lid and lid in scraped_ids:
                continue

            if scrape_details and listing.get("url"):
                det, img_urls = parse_detail(listing["url"])
                listing.update(det)
                if download_imgs and lid and img_urls:
                    listing["images_downloaded"] = download_images(img_urls, lid)
                else:
                    listing["images_downloaded"] = 0
                time.sleep(DETAIL_DELAY)

            all_listings.append(listing)

        # Save after every page
        new_df = pd.DataFrame(all_listings)
        combined = pd.concat([existing_df, new_df], ignore_index=True) if existing_df is not None else new_df
        combined.drop_duplicates(subset="listing_id", keep="last", inplace=True)
        combined.to_csv(output_file, index=False, encoding="utf-8-sig")
        logger.info(f"  {len(combined)} total listings saved.")

        if page >= effective_max:
            break
        page += 1
        time.sleep(PAGE_DELAY)

    logger.info(f"Done! CSV → {output_file}")
    return combined if 'combined' in dir() else pd.DataFrame()

In [20]:
diagnose()


=== DIAGNOSIS MODE ===
Cards found: 27

--- Card 1 ---
  title: Honda Civic Rebirth 2014 Oriel Prosmatec UG for Sale
  url: https://www.pakwheels.com/used-cars/honda-civic-2014-for-sale-in-lahore-11261523
  listing_id: 11261523
  price: PKR 36.75lacs
  location: Lahore
  year: 2014
  mileage: 125,000 km
  fuel_type: Petrol
  engine_cc: 1800 cc
  transmission: Automatic
  last_updated: Updated less than a minute ago

--- Card 2 ---
  title: Lexus Nx  2016 300H Luxury for Sale
  url: https://www.pakwheels.com/used-cars/lexus-nx-2016-for-sale-in-islamabad-10972978
  listing_id: 10972978
  price: PKR 1.89crore
  location: Islamabad
  year: 2016
  mileage: 91,660 km
  fuel_type: Hybrid
  engine_cc: 2500 cc
  transmission: Automatic
  last_updated: Updated 1 minute ago

--- Card 3 ---
  title: Hyundai Sonata  2025 N Line 2.5 Turbo for Sale
  url: https://www.pakwheels.com/used-cars/hyundai-sonata-2025-for-sale-in-islamabad-11152052
  listing_id: 11152052
  price: PKR 1.69crore
  location: I

In [21]:
# Full scrape — all pages, all details, all images
df = scrape_pakwheels(
    max_pages=None,       # None = all pages (will take several hours)
    download_imgs=True,
    scrape_details=True,
    output_file="pakwheels_vehicles.csv",
    resume=True,          # safe to stop and restart anytime
)

print(f"\nFinal dataset: {df.shape[0]} listings × {df.shape[1]} columns")
print(df[["title", "price", "location", "year", "mileage", "fuel_type",
          "engine_cc", "transmission", "spec_color", "spec_registered_in",
          "spec_body_type", "image_count"]].head(10))

2026-03-15 15:59:32,124 [INFO] ── Page 1/? ──


2026-03-15 15:59:34,234 [INFO] Total pages: 5 | Scraping: 5
2026-03-15 16:06:09,467 [WARNING] Image failed (https://cache3.pakwheels.com/ad_pictures/1400/daihatsu-cast-sport-turbo-sa-iii-2022-140084269.webp): HTTPSConnectionPool(host='cache3.pakwheels.com', port=443): Read timed out. (read timeout=15)
2026-03-15 16:06:26,147 [WARNING] Image failed (https://cache2.pakwheels.com/ad_pictures/1400/daihatsu-cast-sport-turbo-sa-iii-2022-140084268.webp): HTTPSConnectionPool(host='cache2.pakwheels.com', port=443): Read timed out. (read timeout=15)
2026-03-15 16:06:49,456 [WARNING] Image failed (https://cache2.pakwheels.com/ad_pictures/1400/suzuki-cultus-vxri-euro-ii-2015-140076189.webp): HTTPSConnectionPool(host='cache2.pakwheels.com', port=443): Read timed out. (read timeout=15)
2026-03-15 16:12:36,186 [WARNING] Image failed (https://cache3.pakwheels.com/ad_pictures/1400/mazda-323-1986-140083940.webp): HTTPSConnectionPool(host='cache3.pakwheels.com', port=443): Max retries exceeded with url: 

KeyboardInterrupt: 